<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🎯 Lesson 4 精华 — Research Supervisor(多 Agent)

**一句话定位**:**复杂请求拆给多个子 agent 并行做** —— supervisor 用 **隔离的 context window** 协调,避免单 agent 在多子主题任务上的 context 污染。**关键不是「多 agent 越多越好」,是 supervisor 知道何时该派、何时不派**。

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**🤔 上一节单 agent 已经够强了,为啥还要 multi-agent?**

单 agent 在 **简单任务** 上很好。但 **复杂请求**(多个子主题)时会出问题:

| 请求 | 单 agent 的问题 |
|------|-----------------|
| 「**对比 A、B、C 三家公司的 AI 安全做法**」 | 一个 context 要同时存 3 家公司的工具反馈 → **token 爆炸 + context 互相干扰** |
| 「**研究 5 个不同行业的 AI 应用**」 | 同上,**还要全部塞进一个 LLM 调用** |
| 「**列出 100 个 SF 餐厅**」 | 反而 **不需要** multi-agent(单线就够) |

[Anthropic 博客](https://www.anthropic.com/engineering/built-multi-agent-research-system) 描述了这种现象,**[「context clash」](https://www.dbreunig.com/2025/06/22/how-contexts-fail-and-how-to-fix-them.html)**:context window 累积多子主题 tool 调用时,**关键信息互相干扰、模型分心**。

**解药**:**multi-agent 系统** —— 每个子主题给一个子 agent,**子 agent 有隔离的 context window**。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🏗️ Supervisor 架构 — 派单 + 隔离

</div>

```
Research Brief
      │
      ▼
┌──────────────────────┐
│   Supervisor LLM     │ ◀────────────────────┐
│   决定:             │                       │
│   - 拆几个子任务?    │                       │
│   - 各子任务给谁?    │                       │
└────┬─────────────────┘                       │
     │                                          │
     │ ConductResearch(子主题 A)                │
     │ ConductResearch(子主题 B)                │ 整合
     │ ConductResearch(子主题 C)                │ findings
     ▼                                          │
  ┌─────────┐ ┌─────────┐ ┌─────────┐          │
  │ 子 agent│ │ 子 agent│ │ 子 agent│          │
  │   A     │ │   B     │ │   C     │          │
  │(隔离    │ │(隔离    │ │(隔离    │          │
  │ context)│ │ context)│ │ context)│          │
  └────┬────┘ └────┬────┘ └────┬────┘          │
       │           │           │                │
       └───────────┼───────────┘                │
                   │                            │
                   ▼ 各自返回 notes              │
          ┌──────────────────┐                  │
          │ Supervisor 综合  │ ─────────────────┘
          │ + 决定要不要补搜  │
          └──────────────────┘
                   │
                   ▼
                  END(notes 准备好,可以写报告)
```

| 角色 | 职责 |
|------|------|
| **Supervisor** | **派单 + 整合**(不亲自做研究) |
| **Sub-agent** | **独立做研究**(每个都是上一节学的 research agent) |
| **ConductResearch** 工具 | supervisor 派单的入口(调用它 = 起一个子 agent) |
| **think_tool** | supervisor **自己反思**(派多少 / 该不该停) |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 核心机制:context isolation(上下文隔离)**

每个子 agent 调用时,**它的 messages 是全新的**(只接收 supervisor 给的描述),**看不到** 其他子 agent 的工作。

好处:

- ✅ **子 agent A 搜的 token 大堆,不污染子 agent B**
- ✅ **每个子 agent 都聚焦自己的小任务**
- ✅ supervisor 收到的是 **每个子 agent 的精炼总结**,而不是原始 trail

→ 这就是 **「**多个 context window 并行 > 一个大 context window**」** 的本质。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 📏 Supervisor 的 4 条 Prompt 准则

</div>

Supervisor 的 prompt 设计 **类似上一节的 research agent**,但 **重点在派单决策**:

### 🅰️ ① 像 agent 一样思考

| 指令 | 意图 |
|------|------|
| **仔细读 brief** | 锁定要研究啥 |
| **决定怎么派单** | 有没有 **可并行的独立方向**? |
| **每次 ConductResearch 后暂停评估** | 还缺啥? |

### 🅱️ ② 具体启发(派单的硬限制)

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 「偏好单 agent」是反直觉的关键准则**

| 限制 | 数值 / 原因 |
|------|-----------|
| **偏好单 agent** | **除非有明确并行机会,默认用单 agent** |
| **能自信回答就停** | 别为了完美无限派单 |
| **ConductResearch 上限 3 次** | 找不到就停 |

**为啥反直觉?** 因为 multi-agent 听起来很酷,**新手往往滥用**。但 multi-agent 的代价:

- 🔥 **token 成本 N 倍**(N 个子 agent + 1 个 supervisor)
- ⏱️ **延迟 ≥ 最慢的子 agent**
- 🐛 **协调失败模式更多**(子 agent 跑偏、supervisor 整合不好)

**结论**:**multi-agent 是 last resort,不是默认选择**。

</div>

### 🅲 ③ 展示思考(think_tool)

**派单前**:能拆吗?

**派单后**:
- 找到啥关键信息?
- 还缺什么?
- 够全面回答了吗?
- 继续搜还是给答案?

### 🅳 ④ 派单规模规则

| 任务类型 | 子 agent 数 | 例子 |
|---------|-----------|------|
| **简单事实查找 / 列表 / 排名** | **1 个** | 「列出 SF top 10 咖啡馆」 |
| **明确对比任务**(N 项) | **N 个** | 「对比 OpenAI / Anthropic / DeepMind 的 AI 安全」→ 3 个 |
| **多维度独立分析** | 每维度 1 个 | 「评估某公司的 财务 + 战略 + 团队」→ 3 个 |

**关键**:**子任务必须"清晰、独立、不重叠"**。如果两个子任务需要互相参考,**就不该拆**。

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧠 ConductResearch 工具 — 派单的入口

</div>

Supervisor 不直接做研究,它通过调用 **`ConductResearch`** 工具来派单。

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">@tool
class ConductResearch(BaseModel):
    """Delegate research to a sub-agent."""
    <span style="background:#3d3414; color:#fde047; padding:0 2px;">research_topic: str</span> = Field(
        description="具体的子主题(对子 agent 而言要清晰、独立、可执行)"
    )
</code></pre>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 `ConductResearch` 是个**特殊的工具**

看起来就是个普通 tool 调用,但 **执行时实际起了一个完整的子 agent**(上一节的 research_agent):

```python
@supervisor_tools.add
async def conduct_research(research_topic: str):
    # 起子 agent,messages 是隔离的
    result = await research_agent.ainvoke({
        "messages": [HumanMessage(research_topic)]
    })
    # 只回传 compressed_research(不是整个 trail)
    return result["compressed_research"]
```

**子 agent 完成后,只回传 `compressed_research`** 给 supervisor。

→ supervisor 看到的是 **精炼总结**,不是子 agent 的几千字搜索 trail。**这就是 context isolation 在物理层面的体现**。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧪 Eval — 测 supervisor 的派单判断

</div>

本节 eval 的核心:**确认 supervisor 「**该并行时才并行**」**。

### 📊 两个测试场景

| 测试 | 输入 | 期望 supervisor 行为 |
|------|------|---------------------|
| **例 1**:对比 3 家 AI 实验室 | 「Compare OpenAI vs Anthropic vs DeepMind」 | **并行派 3 个子 agent** |
| **例 2**:Chelsea top 餐厅 | 「Top restaurants in Chelsea Manhattan」 | **单线研究** |

**判断准则**:

- ✅ **对比 / 比较多个独立项** → 派多个
- ❌ **列表 / 排名 / 推荐** → 单线就够

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**💡 为啥排名任务不该派多个?**

「**列出 top 10 咖啡馆**」——**没法把「top 10」拆给 10 个子 agent**,因为:

- 子 agent 不知道「top 10」的标准(评分?评论数?)
- 子 agent 之间 **可能重复研究** 同一家
- 综合阶段难做(怎么排序?)

**只有「**子任务清晰、独立、不重叠**」时,multi-agent 才有意义**。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🎭 类比 — Supervisor 像什么?

</div>

### 🅰️ 类比:**项目经理 vs 程序员**

好的项目经理 **不写代码**,他们:

- 把大任务 **拆成小任务**
- **派给合适的人**(前端、后端、设计师)
- 收集各人成果 **整合成项目交付**
- 知道 **何时该派、何时该自己上**(简单任务别折腾)

**Supervisor = 项目经理**;**Sub-agents = 程序员**。

### 🅱️ 类比:**Manus 的 Subagents 设计**

[Manus 团队的博客](https://manus.im/blog/Context-Engineering-for-AI-Agents-Lessons-from-Building-Manus) 提了相同模式 — **「**治乱**」**:

> 「context 越长,agent 越乱。**派单给子 agent**,主线只看精炼总结,避免被细节淹没。」

→ supervisor 派单本质是 **「**治乱**」** —— 用 **隔离的 context window** 当 **天然的「忘事剂」**。

### 🆚 对比:**单 agent vs Multi-agent**

| 维度 | 单 agent | Multi-agent |
|------|---------|-------------|
| **简单任务**(单一目标) | ⚡ 快、便宜 | 🐢 过度工程,反而慢 |
| **复杂多子主题** | ❌ context 混乱 | ✅ 隔离 context,质量提升 |
| **token 成本** | 💰 1× | 💰💰💰 N+1× |
| **延迟** | ⏱️ 顺序累加 | ⏱️ ≥ 最慢子 agent |
| **debug 难度** | 🟢 简单 | 🔴 多 agent 协调失败模式多 |

**判断口诀**:**「**能不派,就不派**」**。复杂请求里,**先问自己**:「**这真的需要多个独立子任务吗?**」

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**⚠️ 5 个最容易踩的坑**

1. **❌ 默认所有任务都派 multi-agent** → token 暴增 3-5×,**还没单 agent 效果好**

2. **❌ 子任务有重叠** → 子 agent 各自重复工作,supervisor 整合时还要去重

3. **❌ 子任务描述太宽泛**(如「研究 X」)→ 子 agent 不知道边界,**搜了一堆无关内容**

4. **❌ 没把 `compressed_research` 隔离回传** → 子 agent 整个 messages 倒灌进 supervisor 的 context → **失去 isolation 意义**

5. **❌ supervisor prompt 没有「偏好单 agent」准则** → LLM 喜欢「**多就是好**」,会过度派单

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

## ✨ 一句话带走

**Multi-Agent 的核心 = 「**用多个隔离 context 替代一个大 context**」**:

- **架构**:supervisor 派单 + sub-agent 独立做 + supervisor 整合
- **关键能力**:context isolation —— 子 agent 只回传 `compressed_research`
- **「**偏好单 agent**」** 是反直觉但正确的准则
- **派单规则**:**清晰、独立、不重叠** 的子任务才适合并行

### 🎯 学完这节,你应该能回答:

1. ✅ **什么时候用 multi-agent?**(多子主题 + 子任务独立)
2. ✅ **什么时候不用?**(列表 / 排名 / 简单事实查找)
3. ✅ **multi-agent 的代价是什么?**(token N× / 延迟 ≥ 最慢子 / debug 难)
4. ✅ **context isolation 怎么物理实现?**(子 agent messages 隔离 + 只回传 compressed)
5. ✅ **Supervisor 怎么避免过度派单?**(prompt 写「**偏好单 agent**」+ 上限 3 次 + think_tool 反思)

📎 配套阅读:[`4_research_supervisor.ipynb` 主课](./4_research_supervisor.ipynb) · 下一节 [`5_z_⭐️_本节精华_FullSystem.ipynb`](./5_z_⭐️_本节精华_FullSystem.ipynb)

</div>